# Phase 4A: Class-Imbalance Handling & Training Preparation
### Project: Automated Classification of Martian Surface Images Captured by NASA's Curiosity Rover Using Machine Learning

This notebook verifies and demonstrates the class-imbalance strategy prepared for the 5 classical ML models:
1. **Actual Data Verification**: Loading `y_train`, `y_val`, `y_test` and inspecting feature dimensions from Phase 3 artifacts.
2. **Natural Imbalance Audit**: Verifying class distributions across all 25 classes and the extreme ~122x imbalance ratio.
3. **Balanced Class Weight Calculation**: Demonstrating that weights are computed strictly from `y_train` using $\text{weight}(c) = \frac{N}{K \times N_c}$.
4. **Model API Support Matrix**: Documenting estimator support for `class_weight` across KNN, Naive Bayes, Decision Tree, Random Forest, and SVM.
5. **Architectural Leakage-Safe Scaling**: Verifying `TrainingScaler` fits exclusively on `X_train` without data leakage.
6. **PCA Strategy**: Preserving PCA as an unapplied, controlled experiment for Phase 4B.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Ensure project root is on sys.path
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import config
from models.imbalance import (
    audit_class_distribution,
    compute_balanced_class_weights,
    get_model_imbalance_support,
    validate_class_weights
)
from models.scaling import TrainingScaler, PCAStrategy

print(f"Project root: {config.PROJECT_ROOT}")
print(f"Features dir: {config.FEATURES_DIR}")

## 1. Actual Class Distribution Audit from Phase 3 Artifacts
Load actual labels from disk and audit class representation.

In [ ]:
y_train = np.load(config.FEATURES_DIR / "y_train.npy")
y_val = np.load(config.FEATURES_DIR / "y_val.npy")
y_test = np.load(config.FEATURES_DIR / "y_test.npy")

class_names = {}
with open(config.CLASS_MAPPING_PATH, "r") as f:
    for line in f:
        parts = line.strip().split(None, 1)
        if len(parts) == 2 and parts[0].isdigit():
            class_names[int(parts[0])] = parts[1].strip()

df_audit = audit_class_distribution(y_train, y_val, y_test, class_names)
df_audit

## 2. Balanced Class Weight Calculation (y_train ONLY)
Calculated using $\text{weight}(c) = \frac{N}{K \times N_c}$ exclusively from training labels.

In [ ]:
weights = compute_balanced_class_weights(y_train)
validate_class_weights(weights, y_train)

df_weights = pd.DataFrame([
    {
        "Class ID": cid,
        "Class Name": class_names[cid],
        "Train Count": int(np.sum(y_train == cid)),
        "Balanced Weight": round(weights[cid], 4)
    }
    for cid in sorted(weights.keys())
])
df_weights

## 3. Model Imbalance Support Matrix
Inspect scikit-learn model capabilities for class weighting.

In [ ]:
support = get_model_imbalance_support()
df_support = pd.DataFrame([
    {
        "Model": mname,
        "Supports class_weight": "YES" if minfo["supports_class_weight"] else "NO",
        "Imbalance Mechanism": minfo["supported_mechanism"],
        "Baseline Config": str(minfo["baseline_config"]),
        "Balanced Config": str(minfo["balanced_config"])
    }
    for mname, minfo in support.items()
])
df_support

## 4. Leakage-Safe TrainingScaler Verification
Architecturally verifying that `fit()` only accepts training data.

In [ ]:
with np.load(config.FEATURES_DIR / "X_train.npz") as d:
    X_train = d["X"]

scaler = TrainingScaler()
scaler.fit(X_train, split_name="train")
print("Provenance report:", scaler.get_provenance_report())

X_train_scaled = scaler.transform(X_train[:10], split_name="sample")
print("Scaled sample shape:", X_train_scaled.shape, "finite:", np.isfinite(X_train_scaled).all())